In [ ]:
!git clone https://github.com/YPolina/Medicine.git

In [1]:
%cd ./Medicine/BELKA/training

/content/Medicine/BELKA/training


In [ ]:
!pip install -r ../requirements.txt

In [1]:
import sys
import h5py
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))

sys.modules.pop("functionality.models", None)
sys.modules.pop("functionality.data_preparation", None)
from functionality.data_preparation import IterableEmbDataset, train_model
from functionality.models import ChemBertaBinaryClassifierLightning

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from transformers import AutoTokenizer, AutoModel

import torch
import pickle
from tqdm import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast

/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train/val sets for each of the protein

In [19]:
binds_0 = pd.read_parquet("../intermediates/downsampled_0_50_mln")
binds_1 = pd.read_parquet("../intermediates/1_class")
final_data = pd.concat([binds_0, binds_1], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

del binds_1
del binds_0

protein_names = final_data.protein_name.unique()
save_dir = '../intermediates/train_data'

for protein_name in protein_names:
    protein_data = final_data[final_data.protein_name == protein_name]

    train_data, val_data = train_test_split(
        protein_data, test_size=0.1, random_state=42, shuffle=False
    )
    train_path = os.path.join(save_dir, protein_name, f"{protein_name}_train.parquet")
    val_path = os.path.join(save_dir, protein_name, f"{protein_name}_val.parquet")

    os.makedirs(os.path.dirname(train_path), exist_ok=True)
    os.makedirs(os.path.dirname(val_path), exist_ok=True)
    
    train_data.to_parquet(train_path)
    val_data.to_parquet(val_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
def compute_and_save_embeddings(model_name, smiles, labels, save_path, batch_size=1000):
    """
    Compute embeddings for a list of SMILES strings in batches and save them efficiently using HDF5
    
    Args:
        model_name (str): Pretrained model name
        smiles (pd.Series): Data containing SMILES strings
        labels (pd.Series): Corresponding labels
        save_path (str): Path to save computed embeddings and labels
        batch_size (int): Number of SMILES strings to process per batch
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    with h5py.File(save_path, "w") as h5f:
        dset_embeddings = h5f.create_dataset("embeddings", shape=(0, 768), maxshape=(None, 768), dtype=np.float32, compression="gzip")
        dset_labels = h5f.create_dataset("labels", shape=(0,), maxshape=(None,), dtype=np.int64, compression="gzip")

        for i in tqdm(range(0, len(smiles), batch_size), desc="Computing Embeddings"):
            batch_smiles = smiles.iloc[i : i + batch_size].tolist()
            batch_labels = labels.iloc[i : i + batch_size].values.astype(np.int64)

            tokens = tokenizer(batch_smiles, padding=True, truncation=True, max_length=512, return_tensors="pt")
            tokens = {k: v.to(device) for k, v in tokens.items()}

            with torch.no_grad(), autocast():
                outputs = model(**tokens)

            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            dset_embeddings.resize(dset_embeddings.shape[0] + batch_embeddings.shape[0], axis=0)
            dset_embeddings[-batch_embeddings.shape[0]:] = batch_embeddings

            dset_labels.resize(dset_labels.shape[0] + batch_labels.shape[0], axis=0)
            dset_labels[-batch_labels.shape[0]:] = batch_labels

            # Memory cleanup
            del batch_smiles, batch_labels, tokens, outputs, batch_embeddings
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Embeddings and labels saved to {save_path}")
    return save_path

In [3]:
protein_names = ['sEH', 'BRD4', 'HSA']
if os.getenv('WORKING_ENV') == 'colab':
    save_dir = '/content/drive/MyDrive/embeddings/'
else:
    save_dir = '../intermediates/embeddings/'
models = {
    "ChemBert": "seyonec/PubChem10M_SMILES_BPE_450k",
    "MolFormer": "ibm/MoLFormer-XL-both-10pct"
}
batch_size=1000

for model_name, model_ in models.items():
    for protein_name in protein_names:
        print(f"Embeddings calculations for protein: {protein_name}")

        train_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_train.parquet')
        val_data = pd.read_parquet(f'../intermediates/train_data/{protein_name}/{protein_name}_val.parquet')
        
        train_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_train_embeddings.h5")
        val_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_val_embeddings.h5")

        compute_and_save_embeddings(model_, train_data["molecule_smiles"], train_data['binds'],  train_embeddings_path, batch_size)
        compute_and_save_embeddings(model_, val_data["molecule_smiles"], val_data['binds'], val_embeddings_path, batch_size)


Embeddings calculations for protein: sEH


Computing Embeddings:   0%|          | 0/3648 [00:00<?, ?it/s]/tmp/ipykernel_136374/2722948200.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Computing Embeddings:   0%|          | 2/3648 [01:25<43:05:35, 42.55s/it]


KeyboardInterrupt: 

In [12]:
def train_model(model_name, protein_names, emb_path="../intermediates/embeddings"):
    for protein_name in protein_names:

        print(f"Training model for protein: {protein_name}")

        train_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_train_embeddings.h5")
        val_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_val_embeddings.h5")
        
        train_dataset = IterableEmbDataset(train_embeddings_path)
        train_loader = DataLoader(train_dataset, batch_size=1000, num_workers=4)

        val_dataset = IterableEmbDataset(val_embeddings_path)
        val_loader = DataLoader(val_dataset, batch_size=1000, num_workers=4)

        logger = CSVLogger("logs", name=model_name)
        early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")
        checkpoint_callback = ModelCheckpoint(
            dirpath="../checkpoints",
            filename=f"{model_name}_{protein_name}-{{epoch}}-{{val_loss:.4f}}",
            monitor="val_loss",
            save_top_k=1,
            mode="min",
            save_last=True,
            verbose=True,
        )

        trainer = pl.Trainer(
            max_epochs=20,
            accelerator="auto",
            devices=1,
            log_every_n_steps=2,
            callbacks=[early_stopping, checkpoint_callback],
            logger=logger,
        )


        chemberta_model = ChemBertaBinaryClassifierLightning()
        trainer.fit(chemberta_model, train_loader, val_loader)

        os.makedirs("../intermediates/models", exist_ok=True)
        trainer.save_checkpoint(f"../intermediates/models/{model_name}_{protein_name}.ckpt")

        print(f"Completed training for protein: {protein_name}")

        del train_data, val_data, train_dataset, val_dataset, train_loader, val_loader, chemberta_model
        gc.collect()

In [13]:
train_model('ChemBert', ['HSA'])

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



  | Name    | Type             | Params | Mode 
-----------------------------------------------------
0 | loss_fn | CrossEntropyLoss | 0      | train
1 | auroc   | BinaryAUROC      | 0      | train
2 | dropout | Dropout          | 0      | train
3 | fc1     | Linear           | 98.4 K | train
4 | fc2     | Linear           | 258    | train
-----------------------------------------------------
98.7 K    Trainable params
0         Non-trainable params
98.7 K    Total params
0.395     Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Training model for protein: HSA
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/user/Desktop/Pharm/Medicine/BELKA/venv/lib/python3.10/site-packages/pytorch_lightning/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


Epoch 0: 100%|██████████| 3370/3370 [03:01<00:00, 18.59it/s, v_num=1, val_loss=0.280, val_auc=0.795, train_loss=0.301, train_auc=0.730]

...

Epoch 8: 100%|██████████| 3370/3370 [29:00<00:00,  1.94it/s, v_num=1, val_loss=0.249, val_auc=0.847, train_loss=0.250, train_auc=0.843]

Epoch 8, global step 30330: 'val_loss' reached 0.24898 (best 0.24898), saving model to '/home/user/Desktop/Pharm/Medicine/BELKA/checkpoints/ChemBert_HSA-epoch=8-val_loss=0.2490.ckpt' as top 1


Epoch 9:  81%|████████  | 2732/3370 [22:26<05:14,  2.03it/s, v_num=1, val_loss=0.249, val_auc=0.847, train_loss=0.250, train_auc=0.843]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
train_model('ChemBert', ['BRD4'])